# WeMM-Embedding-2B · 视频文字搜索 MVP

用 [tencent/WeMM-Embedding-2B](https://huggingface.co/tencent/WeMM-Embedding-2B) 实现：**加载一个视频 → 输入文字描述 → 定位到最匹配的时间段并播放**。

流程：
1. 跑通官方示例（文本 / 图片 / 视频 → 2048 维向量）
2. 把视频切成固定长度的小片段（默认 8 秒）
3. 每个片段编码成向量，文字 query 编码成向量
4. 余弦相似度排序 → 得到时间段 → 内嵌播放器播放命中片段

> **运行前**：菜单 `代码执行程序 → 更改运行时类型 → T4 GPU`。免费 T4（16GB 显存）够用。

In [ ]:
# 确认 GPU 可用
!nvidia-smi

In [ ]:
# 安装依赖（约 2-3 分钟）。版本按模型卡要求固定。
# torchcodec + av：新版 torchvision 移除了 read_video，sentence-transformers 读视频需要 torchcodec 后端（本地 Mac 验证时踩过的坑）
!pip install -q "transformers==5.2.0" "qwen-vl-utils[decord]==0.0.14" "sentence-transformers>=5.7.0" "accelerate>=1.1.0" torchcodec av

## 1. 加载模型

T4 是 Turing 架构，不支持 bfloat16，自动降级到 float16；A100/L4 等 Ampere+ 用 bfloat16。首次运行会下载约 6GB 权重。

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "没有检测到 GPU，请切换运行时类型"

cc_major = torch.cuda.get_device_capability()[0]
dtype = torch.bfloat16 if cc_major >= 8 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)}, dtype: {dtype}")

model = SentenceTransformer(
    "tencent/WeMM-Embedding-2B",
    trust_remote_code=True,
    device="cuda",
    model_kwargs={"dtype": dtype},
)
print("模型加载完成")

## 2. 跑通官方示例

模型卡上的原始示例：两条文字 query 对三个文档（纯文本 / 图片 / 视频）算相似度。
预期：第 1 条 query（Llama 4）应命中图片文档，第 2 条（麻婆豆腐做法）应命中文本和视频文档。

In [ ]:
queries = [
    "Which Llama 4 model variants are available?",
    "How is mapo tofu prepared?",
]
documents = [
    "Mapo tofu is a Sichuan dish of soft tofu simmered in a spicy, numbing sauce.",
    {
        "image": "https://huggingface.co/datasets/sentence-transformers/example-documents/resolve/main/llama4_hgf.png",
        "text": "Represent this image.",
    },
    {
        "video": "https://huggingface.co/datasets/sentence-transformers/example-documents/resolve/main/mapo_tofu.mp4",
        "text": "Represent this video.",
    },
]

query_embeddings = model.encode_query(queries)
document_embeddings = model.encode_document(documents)

similarities = model.similarity(query_embeddings, document_embeddings)
print("相似度矩阵（行=query，列=文本/图片/视频）:")
print(similarities)

## 3. 准备测试视频

默认下载开源短片 Big Buck Bunny 并截取前 3 分钟做演示。

**换成自己的视频**：左侧文件面板上传后，把 `VIDEO_PATH` 改成你的文件路径，重跑本节及之后的 cell 即可。

In [ ]:
import os

VIDEO_PATH = "demo_source.mp4"   # 换成自己的视频时改这里
DEMO_DURATION = 180              # 只取前 N 秒做演示，None 表示整段

# Google GCS 的示例桶已关闭匿名访问，改用 Blender 官方源（zip 包）
if not os.path.exists(VIDEO_PATH):
    !wget -q -O bbb.zip "https://download.blender.org/peach/bigbuckbunny_movies/BigBuckBunny_640x360.m4v.zip"
    !unzip -o -q bbb.zip && rm bbb.zip
    trim = f"-t {DEMO_DURATION}" if DEMO_DURATION else ""
    !ffmpeg -y -loglevel error -i BigBuckBunny_640x360.m4v {trim} -c copy {VIDEO_PATH}

!ffprobe -v error -show_entries format=duration -of csv=p=0 {VIDEO_PATH}

## 4. 切片：把视频切成固定长度的片段

重新编码并强制在每 `SEGMENT_SECONDS` 处打关键帧，保证切割点精确（`-c copy` 只能在原关键帧处切，时间会漂移）。同时缩到 360p，加快后续解码和编码速度。

In [ ]:
import glob

SEGMENT_SECONDS = 8
CLIP_DIR = "clips"

!rm -rf {CLIP_DIR} && mkdir -p {CLIP_DIR}
!ffmpeg -y -loglevel error -i {VIDEO_PATH} \
    -vf scale=-2:360 -c:v libx264 -preset veryfast -crf 28 -c:a aac \
    -force_key_frames "expr:gte(t,n_forced*{SEGMENT_SECONDS})" \
    -f segment -segment_time {SEGMENT_SECONDS} -reset_timestamps 1 \
    {CLIP_DIR}/clip_%04d.mp4

clip_paths = sorted(glob.glob(f"{CLIP_DIR}/clip_*.mp4"))
clips = [
    {"path": p, "start": i * SEGMENT_SECONDS, "end": (i + 1) * SEGMENT_SECONDS}
    for i, p in enumerate(clip_paths)
]
print(f"共 {len(clips)} 个片段，每段 {SEGMENT_SECONDS} 秒")

## 5. 给每个片段编码向量

T4 上每段约 1-3 秒，3 分钟视频（约 23 段）大概一分钟跑完。向量只需算一次，之后任意 query 都是毫秒级检索。

In [ ]:
clip_docs = [{"video": c["path"], "text": "Represent this video."} for c in clips]

clip_embeddings = model.encode_document(
    clip_docs,
    batch_size=1,          # 视频输入显存占用大，T4 上保守用 1
    show_progress_bar=True,
)
print("向量矩阵形状:", clip_embeddings.shape)  # (片段数, 2048)

## 6. 文字搜索 → 定位时间段 → 播放

输入一句描述，返回最匹配的时间段并内嵌播放命中片段。

In [ ]:
import base64
from IPython.display import HTML, display

def fmt_time(sec):
    return f"{int(sec) // 60:02d}:{int(sec) % 60:02d}"

def play_clip(path):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video width="480" controls autoplay muted '
        f'src="data:video/mp4;base64,{b64}"></video>'
    ))

def search(query, top_k=3, play=True):
    q_emb = model.encode_query([query])
    scores = model.similarity(q_emb, clip_embeddings)[0]
    ranked = scores.argsort(descending=True)[:top_k]
    print(f'Query: "{query}"')
    for rank, idx in enumerate(ranked, 1):
        c = clips[int(idx)]
        print(f"  #{rank}  {fmt_time(c['start'])} - {fmt_time(c['end'])}  "
              f"score={scores[idx]:.4f}")
    if play:
        best = clips[int(ranked[0])]
        print(f"\n▶ 播放最佳匹配片段 {fmt_time(best['start'])} - {fmt_time(best['end'])}")
        play_clip(best["path"])
    return ranked

In [ ]:
# Big Buck Bunny 的示例 query，换自己的视频后改成对应内容的描述
search("a giant rabbit comes out of its burrow and stretches")

In [ ]:
search("a butterfly flying near flowers")

In [ ]:
# 中文 query 也可以直接试（模型是多语言的）
search("三只小动物站在树上看着下面")

## 7. 调优方向（跑通之后再看）

- **片段长度**：`SEGMENT_SECONDS` 越短定位越精，但片段数和编码时间线性增加；8 秒是精度/速度的合理起点，动作类内容可试 4-5 秒。
- **滑动窗口**：固定切片会把一个动作切在两段之间，可改成带 50% 重叠的窗口（步长 = 长度/2）减少漏检。
- **规模化**：视频多了以后把向量存进 FAISS / Milvus / pgvector，query 时只做一次文本编码 + ANN 检索。
- **部署**：模型卡支持 vLLM 0.27.0 / SGLang 0.5.9 起服务，编码吞吐比 transformers 高不少。
- **本地跑（已在 M5 Max 上验证通过）**：仓库里的 `verify_local.py` 就是本 notebook 的本地版，`device="mps"` + float16。Mac 上需要 `brew install ffmpeg` 并在运行时加 `DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib`（torchcodec 找 ffmpeg 动态库用）。实测 23 段编码仅 26 秒（1.1 秒/段），本地开发完全够用。